# 14 — Pattern viewer

Chart-pattern **shapes** and **S/R zones** on one price chart. Click a name in the legend to show or hide it.

Each pattern draws four things, all under its single legend entry:

| drawn | meaning |
|---|---|
| solid line + dots | the shape's **pivots**, in time order — the outline |
| dashed lines | the **upper and lower trendlines** through those pivots |
| amber line | the **neckline** that had to break for the pattern to count |
| dotted line | the **measured-move target** (pattern height projected from the break) |

**Nothing here is drawn early.** A shape appears only from the bar it *confirmed* — its pivots
were all confirmed beforehand and price had closed through the neckline. That is why a pattern
sometimes looks obvious in hindsight but is drawn later than you would have spotted it by eye:
the chart shows what was actually knowable at the time.

Only section 1 needs editing.

## 1. Config — edit here

In [ ]:
SYMBOL, TF   = "BTCUSDT", "15m"
START, END   = "2024-06-01", "2024-08-01"
SOURCE       = "binance"          # or "dukascopy" with SYMBOL="XAUUSD"

# what to draw
PATTERNS = [
    "symmetrical_triangle",
    "descending_triangle",
    "ascending_triangle",
    "head_shoulders",
]
SHOW_SR_ZONES = True

# which ones are ON when the chart opens (the rest start hidden; toggle from the legend).
# use "sr_zones" for the zones, or set to tuple(PATTERNS) to start with everything on.
START_VISIBLE = ("sr_zones", "head_shoulders")

# --- detector sensitivity -------------------------------------------------------------
ATR_MULT        = 2.0    # zigzag threshold: LOWER = more, smaller swings = more patterns
TOL             = 0.12   # how close two pivots must be to count as "the same level"
LOOKBACK_PIVOTS = 6      # how many recent pivots a shape may span
MAX_WAIT        = 150    # bars a completed shape stays live waiting for its neckline break
EXTEND          = 12     # bars to extend the neckline/target lines past the break

# --- S/R zone sensitivity -------------------------------------------------------------
ZONE_LR      = 10        # pivot look-around each side
CLUSTER_PCT  = 0.15      # merge levels within this % of each other
MIN_TOUCHES  = 2         # hide zones touched fewer times than this

HEIGHT = 820

## 2. Data

In [ ]:
from quant.data import get_ohlcv

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
print(f"{len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")

## 3. What was found

A quick count before plotting — if a pattern shows 0 here, lower `ATR_MULT` or widen the date range.

In [ ]:
from quant.indicators import chart_pattern_shapes

shapes = chart_pattern_shapes(df, patterns=PATTERNS, atr_mult=ATR_MULT, tol=TOL,
                              lookback_pivots=LOOKBACK_PIVOTS, max_wait=MAX_WAIT)
print(shapes.pattern.value_counts().to_string() if len(shapes) else "no patterns found")
shapes[["t", "pattern", "direction", "entry", "stop", "target"]].tail(10)

## 4. Chart — toggle each pattern from the legend

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    df, patterns=PATTERNS, start_visible=START_VISIBLE,
    sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
    cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES,
    atr_mult=ATR_MULT, tol=TOL, lookback_pivots=LOOKBACK_PIVOTS,
    max_wait=MAX_WAIT, extend=EXTEND,
    title=f"{SYMBOL} {TF}  {START} to {END}", height=HEIGHT)
fig

### Open it in a browser (bigger, and it survives the notebook being closed)

In [ ]:
from quant.viz import open_in_browser

open_in_browser(fig)

---
### Zoom to one pattern

Set `PICK` to a row number from section 3 to focus the chart around that occurrence.

In [ ]:
PICK = -1        # -1 = the most recent one
PAD  = 60        # bars of context each side

if len(shapes):
    row = shapes.iloc[PICK]
    lo = max(int(min(row.pivot_i)) - PAD, 0)
    hi = min(int(row.confirm_i) + PAD, len(df) - 1)
    sub = df.iloc[lo:hi + 1].reset_index(drop=True)
    zoom = pattern_figure(sub, patterns=PATTERNS, start_visible=tuple(PATTERNS) + ("sr_zones",),
                          sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
                          cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES,
                          atr_mult=ATR_MULT, tol=TOL, lookback_pivots=LOOKBACK_PIVOTS,
                          max_wait=MAX_WAIT, extend=EXTEND, height=700,
                          title=f"{row.pattern} confirmed {row.t}")
    zoom.show()
else:
    print("nothing to zoom to")

---
**A note on what you are looking at.** These shapes are drawn honestly — from the bar they could
first be known — and the research in `docs/WHAT_WORKS.md` found they do **not** predict direction
(chart patterns resolve their own way only 46–47% of the time). What they *do* predict is size: the
20 bars after one of these confirms cover **12–19% more range** than normal. Read them as "expect a
bigger move", not as "expect this direction".